In [1]:
import scanpy as sc
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import glob
import pandas as pd
import os


def get_anndata_her2st(sample):
    img_path = os.path.join(sample, [f for f in os.listdir(sample) if f.endswith('.jpg') and f != 'spot_view.jpg'][0])
    
    spots_coord = pd.read_csv(os.path.join(sample, "spots.csv"), index_col=0)
    cnt = pd.read_csv(os.path.join(sample, "stdata.csv"), index_col=0)
    
    spots_coord = spots_coord.loc[cnt.index]

    location = np.matrix(spots_coord.values)
    Xdense = np.matrix(cnt.values)
    
    batch = np.array([sample.split('/')[-1] for i in range(Xdense.shape[0])])

    X = csr_matrix(Xdense, dtype=np.float32)
    adata_vis = ad.AnnData(X, obsm={"spatial": location}, obs={"sample": batch})
    adata_vis.obs_names = [sample.split('/')[-1] + '_' + idx for idx in list(spots_coord.index)]
    # adata_vis.obs_names = list(spots_coord.index)
    adata_vis.var_names = list(cnt.columns)
    return adata_vis

samples_path = [path for path in glob.glob("/data1/r20user3/shared_project/Hist2Cell/data/her2st/*") if not path.endswith(".h5ad") or not path.endswith(".ipynb")]

adata_list = []
from tqdm import tqdm
for i in tqdm(range(0, len(samples_path))):
    adata_1 = get_anndata_her2st(samples_path[i])
    adata_list.append(adata_1)

adata_vis = ad.concat(adata_list)
adata_vis

100%|██████████| 36/36 [00:24<00:00,  1.49it/s]
/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/anndata/_core/merge.py:217: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(dtype):


AnnData object with n_obs × n_vars = 13620 × 11871
    obs: 'sample'
    obsm: 'spatial'

In [2]:
import pandas as pd

results_folder = '/data1/r20user3/shared_project/Hist2Cell/data/breast_sc/cell2location_res'
ref_run_name = f'{results_folder}/breast_reference_signatures'
inf_aver = pd.read_csv(f"{ref_run_name}/inf_aver.csv", index_col=0)
inf_aver

,fibroblast,T cell,mast cell,macrophage,vascular associated smooth muscle cell,CD4-positive helper T cell,lymphocyte,natural killer cell,"CD4-positive, alpha-beta T cell",basal cell,...,class switched memory B cell,IgG plasma cell,IgA plasma cell,conventional dendritic cell,endothelial cell of lymphatic vessel,capillary endothelial cell,luminal epithelial cell of mammary gland,mammary gland epithelial cell,vein endothelial cell,endothelial cell of artery
ENSG00000187634,0.016341,0.003520,0.000729,0.000525,0.000805,0.000034,0.001233,0.000295,0.000218,0.000394,...,0.000222,0.006724,0.004025,0.000890,0.000852,0.001395,0.004111,0.000428,0.000224,0.000118
ENSG00000188976,0.117370,0.146280,0.016717,0.073333,0.051991,0.039917,0.051192,0.049764,0.043392,0.171738,...,0.042114,0.026784,0.040397,0.065859,0.053536,0.109784,0.256512,0.276506,0.165131,0.051012
ENSG00000187583,0.000311,0.005345,0.000754,0.003718,0.000774,0.003845,0.008622,0.000531,0.001245,0.003728,...,0.000648,0.021841,0.023673,0.004289,0.000818,0.000226,0.051024,0.063262,0.000308,0.000094
ENSG00000188290,0.102318,0.060894,0.029283,0.107470,1.194010,0.007908,0.039131,0.012359,0.001107,0.012373,...,0.003083,0.004363,0.005516,0.153251,0.294717,0.089528,0.950762,0.654805,0.069212,0.339997
ENSG00000187608,0.148548,0.641984,0.027741,2.256322,0.478522,0.162423,0.235466,0.186748,0.154152,0.004705,...,0.052665,0.103493,0.156098,0.438484,0.182963,0.565770,0.097423,0.059964,0.560094,0.859192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000198695,0.150368,0.914952,0.047736,0.389017,0.104164,0.220048,0.218192,0.131441,0.310187,0.237162,...,0.240536,0.282472,0.556094,0.408530,0.208988,0.410932,1.071885,0.934500,0.439132,0.181709
ENSG00000198727,17.179066,18.372324,9.565456,22.042368,14.920470,6.843974,8.016482,6.811477,11.395451,16.847425,...,10.901145,12.678282,19.543940,21.561310,11.723537,16.444237,33.896400,45.283203,23.438406,9.522370
ENSG00000277836,0.005831,0.948746,1.011197,1.117903,0.377961,0.614520,1.003062,0.849213,0.799271,0.100614,...,1.026887,0.870512,0.867732,1.027124,0.826734,0.537902,0.126450,0.566995,0.468169,0.868281
ENSG00000277856,0.000010,0.913959,0.722972,0.660645,0.000369,0.000196,0.688798,0.242963,0.126896,0.000023,...,0.354873,0.953489,0.528784,0.422856,0.007974,0.000129,0.000016,0.000074,0.000056,0.001172


In [3]:
adata_ref = sc.read_h5ad("/data1/r20user3/shared_project/Hist2Cell/data/breast_sc/cell2location_res/breast_reference_signatures/breast_sc.h5ad")
gene_name_df = pd.DataFrame(data=adata_ref.var['feature_name'].index, index=adata_ref.var['feature_name'])
gene_name_df

,0
feature_name,
SAMD11,ENSG00000187634
NOC2L,ENSG00000188976
PLEKHN1,ENSG00000187583
HES4,ENSG00000188290
ISG15,ENSG00000187608
...,...
MT-ND6,ENSG00000198695
MT-CYB,ENSG00000198727
ENSG00000277836.1,ENSG00000277836


In [4]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, adata_ref.var['feature_name'])

adata_vis = adata_vis[:, intersect].copy()
gene_id = gene_name_df.loc[adata_vis.var_names]
adata_vis.var_names = gene_id[0]
inf_aver = inf_aver.loc[adata_vis.var_names, :].copy()

intersect

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/anndata/_core/anndata.py:1113: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if not is_categorical_dtype(df_full[k]):


array(['A2M', 'A4GALT', 'AAAS', ..., 'ZYX', 'ZZEF1', 'ZZZ3'], dtype=object)

In [5]:
gpu_list = [1]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)

'1'

In [6]:
import cell2location

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver, 
    # the expected average cell abundance: tissue-dependent 
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
) 
mod.view_anndata_setup()

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.j

Anndata setup with scvi-tools version 1.0.4.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │  36   │
│         n_cells          │ 13620 │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 9739  │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                   batch State Registry                   
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃   Source Location   ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['sample'] │     A1     │          0          │
│                     │     A2     │          1          │
│                     │     A3     │          2          │
│                     │     A4     │          3          │
│                     │     A5     │          4          │
│                     │     A6     │          5          │
│                     │     B1     │          6          │
│                     │     B2     │          7          │
│                     │     B3     │          8          │
│                     │     B4     │          9          │
│                     │     B5     │         10          │
│                     │     B6     │         11          │
│                     │     C1     │         12          │
│                     │     C2     │         13          │
│                     │     C3     │         14          │
│                     │     C4     │         15          │
│                     │     C5     │         16          │
│                     │     C6     │         17          │
│                     │     D1     │         18          │
│                     │     D2     │         19          │
│                     │     D3     │         20          │
│                     │     D4     │         21          │
│                     │     D5     │         22          │
│                     │     D6     │         23          │
│                     │     E1     │         24          │
│                     │     E2     │         25          │
│                     │     E3     │         26          │
│                     │     F1     │         27          │
│                     │     F2     │         28          │
│                     │     F3     │         29          │
│                     │     G1     │         30          │
│                     │     G2     │         31          │
│                     │     G3     │         32          │
│                     │     H1     │         33          │
│                     │     H2     │         34          │
│                     │     H3     │         35          │
└─────────────────────┴────────────┴─────────────────────┘

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

In [7]:
from matplotlib import pyplot as plt


mod.train(max_epochs=30000,
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
          use_gpu=True,
         )

# plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training'])

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/scvi/train/_trainrunner.py:76: UserWarning: `use_gpu` is deprecated in v1.0 and will be removed in v1.1. Please use `accelerator` and `devices` instead.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:69: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
L

Epoch 10726/30000:  36%|███▌      | 10725/30000 [26:15<47:52,  6.71it/s, v_num=1, elbo_train=7.19e+7]  

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_vis = mod.export_posterior(
    adata_vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs, 'use_gpu': True}
)

In [ ]:
adata_vis.obsm['q05_cell_abundance_w_sf']

In [ ]:
run_name = f'{results_folder}/stnet'

# Save model
mod.save(f"{run_name}", overwrite=True)
# Save anndata object with results
adata_file = f"{run_name}/sp.h5ad"
adata_vis.write(adata_file)
adata_file

In [ ]:
import scanpy as sc

adata = sc.read(f"{run_name}/sp.h5ad")
adata

In [ ]:
adata.obs['sample']

In [ ]:
from glob import glob

samples_path = [i for i in glob("/data1/r20user3/shared_project/Hist2Cell/data/her2st/*") if not i.endswith('ipynb')]
samples_path

['/data1/r20user3/shared_project/Hist2Cell/data/her2st/B4',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/F3',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A4',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C6',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/G1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/G2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/H1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B3',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/F2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C5',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B5',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C1',
 '/data1/r20user3/shared_project/Hist2Ce

In [ ]:
import scanpy as sc

adata = sc.read('/data1/r20user3/shared_project/Hist2Cell/data/breast_sc/cell2location_res/her2st/sp.h5ad')
adata

AnnData object with n_obs × n_vars = 13620 × 9739
    obs: 'sample', '_indices', '_scvi_batch', '_scvi_labels'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod'
    obsm: 'means_cell_abundance_w_sf', 'q05_cell_abundance_w_sf', 'q95_cell_abundance_w_sf', 'spatial', 'stds_cell_abundance_w_sf'

In [ ]:
adata.obs['sample']

B4_10x14    B4
B4_10x15    B4
B4_10x16    B4
B4_10x17    B4
B4_10x18    B4
            ..
A2_9x26     A2
A2_9x27     A2
A2_9x28     A2
A2_9x29     A2
A2_9x30     A2
Name: sample, Length: 13620, dtype: category
Categories (36, object): ['A1', 'A2', 'A3', 'A4', ..., 'G3', 'H1', 'H2', 'H3']

In [ ]:
from tqdm import tqdm
import os
import pandas as pd

columns_order = pd.read_csv("/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C1/cell_ratio.csv", index_col=0).columns

for sample in tqdm(samples_path):
    idx = sample.split('/')[-1]
    sub_adata = adata[adata.obs['sample'] == idx]
    cell_ratio = sub_adata.obsm['q05_cell_abundance_w_sf']
    index_list = list(cell_ratio.index)
    cell_ratio.index = [x.split('_')[-1] for x in index_list]
    cell_ratio.index.rename('spot_id', inplace=True)
    cell_ratio = cell_ratio[columns_order]
    cell_ratio.to_csv(os.path.join(sample, "cell_ratio_from_raw70m.csv"))

  0%|          | 0/36 [00:00<?, ?it/s]

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/anndata/_core/anndata.py:1113: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if not is_categorical_dtype(df_full[k]):
100%|██████████| 36/36 [00:00<00:00, 36.55it/s]
